In [ ]:
"""
PostgreSQL → Microsoft Fabric Lakehouse

Скрипт выполняет полную загрузку нормализованных финансовых данных
из PostgreSQL в Delta-таблицу Microsoft Fabric Lakehouse.

Этапы:
1. Подключение к PostgreSQL через JDBC.
2. Чтение таблицы transfer_data_to_fabric.
3. Приведение типов данных к схеме Lakehouse.
4. Полная перезапись Delta-таблицы.
5. Проверка результата загрузки.

Важно:
Не храните реальные пароли и адреса серверов в публичном репозитории.
Используйте Fabric Variable Library, Azure Key Vault или placeholders.
"""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


# =========================================================
# CONFIGURATION
# =========================================================

PG_HOST = "<POSTGRES_HOST>"
PG_PORT = "5432"
PG_DATABASE = "<POSTGRES_DATABASE>"
PG_USER = "<POSTGRES_USER>"
PG_PASSWORD = "<POSTGRES_PASSWORD>"

PG_SOURCE_TABLE = "public.transfer_data_to_fabric"
LAKEHOUSE_TARGET_TABLE = "transfer_data_to_fabric"

POSTGRES_DRIVER = "org.postgresql.Driver"


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def to_decimal(column_name: str) -> F.Column:
    """
    Convert a source column to Decimal(38, 18).

    The function:
    - converts the value to string;
    - removes surrounding spaces;
    - removes double quotes;
    - replaces comma decimal separators with dots;
    - converts the result to DecimalType.
    """

    cleaned_value = F.trim(F.col(column_name).cast("string"))
    cleaned_value = F.regexp_replace(cleaned_value, '"', "")
    cleaned_value = F.regexp_replace(cleaned_value, ",", ".")

    return cleaned_value.cast(DecimalType(38, 18))


def read_postgres_table() -> DataFrame:
    """
    Read normalized financial data from PostgreSQL using JDBC.
    """

    jdbc_url = (
        f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
    )

    postgres_query = f"""
    (
        SELECT
            company,
            wallet_bank,
            date,
            depositdate,
            withdrawaldate,
            norm_date,
            pay_id,
            status,
            amount,
            currency,
            commission,
            commissioncurrency,
            exchangerate,
            finalamount,
            project,
            notes,
            reference,
            paymentmethod,
            settlement,
            settlementcomission,
            topups,
            topupscomission,
            affilates,
            affilatescommission,
            chargebacksrefunds,
            chargebacksrefundscommission,
            expenses,
            expensescommission,
            notes222,
            notes333,
            notes444,
            path,
            ingection_id,
            ingection_time,
            raw_hash
        FROM {PG_SOURCE_TABLE}
    ) AS source_data
    """

    return (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", postgres_query)
        .option("user", PG_USER)
        .option("password", PG_PASSWORD)
        .option("driver", POSTGRES_DRIVER)
        .load()
    )


def prepare_lakehouse_dataframe(df_source: DataFrame) -> DataFrame:
    """
    Convert the PostgreSQL DataFrame to the Lakehouse table schema.
    """

    return df_source.select(
        F.col("company").cast("string").alias("company"),
        F.col("wallet_bank").cast("string").alias("wallet_bank"),

        F.col("date").cast("timestamp").alias("date"),
        F.col("depositdate").cast("timestamp").alias("depositdate"),
        F.col("withdrawaldate").cast("timestamp").alias("withdrawaldate"),
        F.col("norm_date").cast("timestamp").alias("norm_date"),

        F.col("pay_id").cast("string").alias("pay_id"),
        F.col("status").cast("string").alias("status"),

        to_decimal("amount").alias("amount"),
        F.col("currency").cast("string").alias("currency"),

        to_decimal("commission").alias("commission"),
        F.col("commissioncurrency")
        .cast("string")
        .alias("commissioncurrency"),

        to_decimal("exchangerate").alias("exchangerate"),
        to_decimal("finalamount").alias("finalamount"),

        F.col("project").cast("string").alias("project"),
        F.col("notes").cast("string").alias("notes"),
        F.col("reference").cast("string").alias("reference"),
        F.col("paymentmethod").cast("string").alias("paymentmethod"),

        to_decimal("settlement").alias("settlement"),
        to_decimal("settlementcomission").alias(
            "settlementcomission"
        ),

        to_decimal("topups").alias("topups"),
        to_decimal("topupscomission").alias(
            "topupscomission"
        ),

        to_decimal("affilates").alias("affilates"),
        to_decimal("affilatescommission").alias(
            "affilatescommission"
        ),

        to_decimal("chargebacksrefunds").alias(
            "chargebacksrefunds"
        ),
        to_decimal("chargebacksrefundscommission").alias(
            "chargebacksrefundscommission"
        ),

        to_decimal("expenses").alias("expenses"),
        to_decimal("expensescommission").alias(
            "expensescommission"
        ),

        F.col("notes222").cast("string").alias("notes222"),
        F.col("notes333").cast("string").alias("notes333"),
        F.col("notes444").cast("string").alias("notes444"),

        F.col("path").cast("string").alias("path"),
        F.col("ingection_id").cast("string").alias(
            "ingection_id"
        ),
        F.col("ingection_time").cast("timestamp").alias(
            "ingection_time"
        ),
        F.col("raw_hash").cast("string").alias("raw_hash")
    )


def write_to_lakehouse(df_ready: DataFrame) -> None:
    """
    Fully overwrite the target Delta table in the Fabric Lakehouse.
    """

    (
        df_ready.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "false")
        .saveAsTable(LAKEHOUSE_TARGET_TABLE)
    )


def validate_lakehouse_load() -> None:
    """
    Display basic validation statistics for the loaded table.
    """

    validation_query = f"""
    SELECT
        wallet_bank,
        COUNT(*) AS rows_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM {LAKEHOUSE_TARGET_TABLE}
    GROUP BY wallet_bank
    ORDER BY rows_count DESC
    """

    spark.sql(validation_query).show(
        100,
        truncate=False
    )


# =========================================================
# MAIN PROCESS
# =========================================================

def main() -> None:
    print("=" * 70)
    print("POSTGRESQL → MICROSOFT FABRIC LAKEHOUSE")
    print("=" * 70)

    print(
        f"Reading PostgreSQL table: {PG_SOURCE_TABLE}"
    )

    df_postgres = read_postgres_table()

    postgres_rows = df_postgres.count()

    print(
        f"Rows read from PostgreSQL: {postgres_rows}"
    )

    print("Preparing Lakehouse schema...")

    df_ready = prepare_lakehouse_dataframe(df_postgres)

    ready_rows = df_ready.count()

    print(
        f"Rows prepared for loading: {ready_rows}"
    )

    if postgres_rows != ready_rows:
        raise RuntimeError(
            "Row count mismatch after data preparation: "
            f"PostgreSQL={postgres_rows}, "
            f"prepared={ready_rows}"
        )

    print("Sample data:")

    display(
        df_ready.limit(20)
    )

    print(
        f"Overwriting Lakehouse table: "
        f"{LAKEHOUSE_TARGET_TABLE}"
    )

    write_to_lakehouse(df_ready)

    lakehouse_rows = spark.table(
        LAKEHOUSE_TARGET_TABLE
    ).count()

    print(
        f"Rows written to Lakehouse: {lakehouse_rows}"
    )

    if ready_rows != lakehouse_rows:
        raise RuntimeError(
            "Row count mismatch after Lakehouse load: "
            f"prepared={ready_rows}, "
            f"Lakehouse={lakehouse_rows}"
        )

    print("Load completed successfully.")

    print("Validation by wallet_bank:")

    validate_lakehouse_load()


main()